# **1) Ingestion pipeline**

In [1]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
# from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

from sentence_transformers import SentenceTransformer

C:\Users\Lapmart\AppData\Local\Temp\ipykernel_12512\4283994956.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [2]:
import transformers
import sentence_transformers

print(transformers.__version__)
print(sentence_transformers.__version__)

5.11.0
5.5.1


### **Configurations**

In [5]:
doc_path="docs"

chunk_size=800
chunk_overlap=0

db_path="db/chroma_db"

### **load_documents**

In [6]:
print(f'Loading documents from {doc_path}')

# document loading
loader = DirectoryLoader(
    path=doc_path,
    glob="*.txt",
    loader_cls=TextLoader, # what type of loader to use for each file > TextLoader = a loader designed for plain .txt files
    loader_kwargs= {"encoding":"utf-8"}, # if we didnt provide this it uses windows defualt encoding cp1252, it cant reads UTF-8 characters (emoji, special symbols, smart quotes)
)

documents = loader.load() # list of lungchain documents

if len(documents) == 0:
    raise FileNotFoundError(f"No .txt files found in {doc_path} directory")

# print("### page_content = ", documents[0].page_content) # contains entire documents here
# print("### metadata = ", documents[0].metadata) # other info of document

''' one file:
Document(
    page_content="...text here...",
    metadata={"source": "file path"}
)
'''

Loading documents from docs


' one file:\nDocument(\n    page_content="...text here...",\n    metadata={"source": "file path"}\n)\n'

In [7]:
for i, doc in enumerate(documents[:2]):
    print("\n###", type(doc))
    print(f"Doc {i+1}")
    print(f"Source : {doc.metadata['source']}")
    print(f"Content length : {len(doc.page_content)} characters\n")


### <class 'langchain_core.documents.base.Document'>
Doc 1
Source : docs\Google.txt
Content length : 232201 characters


### <class 'langchain_core.documents.base.Document'>
Doc 2
Source : docs\Microsoft.txt
Content length : 201014 characters



### **split_documents**

In [8]:
print(f"### Splitting documents into chunks (size={chunk_size})")

text_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

chunks = text_splitter.split_documents(documents=documents) # # list of all chunkns of all files

Created a chunk of size 949, which is longer than the specified 800
Created a chunk of size 922, which is longer than the specified 800
Created a chunk of size 892, which is longer than the specified 800
Created a chunk of size 825, which is longer than the specified 800
Created a chunk of size 921, which is longer than the specified 800
Created a chunk of size 830, which is longer than the specified 800
Created a chunk of size 1055, which is longer than the specified 800
Created a chunk of size 874, which is longer than the specified 800
Created a chunk of size 1436, which is longer than the specified 800
Created a chunk of size 924, which is longer than the specified 800
Created a chunk of size 815, which is longer than the specified 800
Created a chunk of size 1039, which is longer than the specified 800
Created a chunk of size 1078, which is longer than the specified 800
Created a chunk of size 1043, which is longer than the specified 800
Created a chunk of size 880, which is longe

### Splitting documents into chunks (size=800)


In [9]:
if chunks:
    # only first 5 chunks
    for i, chunk in enumerate(chunks[:5]):
        print(f"### Chunk {i+1}")
        print(f"Source : {chunk.metadata}")
        print(f"Chunk length : {len(chunk.page_content)} chars")
        print(f"Chunk content : {chunk.page_content}")
        print("-" * 50)

    if len(chunks) > 5:
        print(f"more {len(chunks) -5} are there...")

### Chunk 1
Source : {'source': 'docs\\Google.txt'}
Chunk length : 600 chars
Chunk content : ﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, computer software,
quantum computing, e-commerce, consumer
electronics, and artificial intelligence (AI).[9] It has
been referred to as "the most powerful company in the The Google logo used since 2015
world" by the BBC[10] and is one of the world's most
valuable brands.[11][12][13] Google's parent company,
Alphabet Inc., is one of the five Big Tech companies
alongside Amazon, Apple, Meta, and Microsoft.
--------------------------------------------------
### Chunk 2
Source : {'source': 'docs\\Google.txt'}
Chunk length : 738 chars
Chunk content : Google was founded on September 4, 1998, by
American computer scientists Larry Page and Sergey
Brin. Together, they own about 14% of its publicly
listed shares an

### **create_vector_store**

#### Download and save embedding model in local folder

In [12]:
# from sentence_transformers import SentenceTransformer

# # Download and save
# model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
# model.save("saved_models/all-mpnet-base-v2")

In [17]:
print("### Creating embeddings and storing them in a vector database")

# embedding_model = OpenAIEmbeddings(model="text-embedding-3-small") # must pay for api key

# open-source embedding model (from online)
# embedding_model = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-mpnet-base-v2", # all-MiniLM-L12-v2, all-MiniLM-L6-v2, all-mpnet-base-v2(768-dim embedding model)
#     model_kwargs={"device":"cpu"},
#     encode_kwargs={"normalize_embeddings":True}
#     )

# load from local
embedding_model = HuggingFaceEmbeddings(
    model_name="saved_models/all-mpnet-base-v2", # all-MiniLM-L12-v2, all-MiniLM-L6-v2, all-mpnet-base-v2
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings":True}
    )

### Creating embeddings and storing them in a vector database


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
# Create chromo vector database
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=db_path,
    collection_metadata={"hnsw:space":"cosine"}
)
print("### vector db created")

### vector db created


In [21]:
# import shutil

# shutil.rmtree(db_path)